# Modelo de riesgo de deserción — Observatorio UCC

Este notebook documenta, paso a paso, el entrenamiento del modelo de **regresión logística** que usa la alerta temprana.
La versión que se lleva a producción la genera `ml/entrenar_modelo.py` (mismas funciones, importadas aquí), que exporta los
coeficientes a `ml/modelo_riesgo.json`. El backend TypeScript (`services/riesgo/ModeloRiesgo.ts`) aplica esos coeficientes:
**no hay servidor Python en producción**.

**Unidad de análisis:** estudiante-semestre. **Objetivo:** `y = 1` si el estudiante desertó al terminar ese semestre o el siguiente
(horizonte de un año académico). Los semestres cuyo desenlace aún no se conoce (censurados) no entran al entrenamiento.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.metrics import roc_auc_score, roc_curve
from entrenar_modelo import nuevo_modelo, metricas, umbrales, dec, pct, DATOS, SEMILLA

df = pd.read_csv(DATOS)
caracteristicas = [c for c in df.columns if c not in ("estudiante_id", "semestre_fila", "y")]
print(f"{len(df)} filas · {df.estudiante_id.nunique()} estudiantes · {len(caracteristicas)} características · tasa base {pct(df.y.mean())}")
df.head()

7949 filas · 1491 estudiantes · 21 características · tasa base 11,4 %


,estudiante_id,semestre_fila,y,promedio,materias_perdidas,asistencia,caida_promedio,horas_trabajo,estrato_1_2,estrato_4_mas,...,fin_beca,fin_credito_ucc,programa:Administración de Empresas,programa:Contaduría Pública,programa:Enfermería,programa:Ingeniería Civil,programa:Ingeniería de Sistemas,programa:Medicina,programa:Odontología,programa:Psicología
0,3171ed86-ce20-4d1d-b50d-8cfd7a5352bc,1,0,3.28,1,80.4,0.00,0,0,0,...,0,0,0,0,0,0,0,0,0,1
1,3171ed86-ce20-4d1d-b50d-8cfd7a5352bc,2,1,3.77,0,81.6,0.00,0,0,0,...,0,0,0,0,0,0,0,0,0,1
2,3171ed86-ce20-4d1d-b50d-8cfd7a5352bc,3,1,2.76,1,87.4,1.01,0,0,0,...,0,0,0,0,0,0,0,0,0,1
3,68366b5e-b3d5-42ad-8c27-b95d45a56102,1,0,3.04,1,86.6,0.00,0,1,0,...,0,0,0,0,0,0,0,0,0,0
4,68366b5e-b3d5-42ad-8c27-b95d45a56102,2,0,3.50,0,76.9,0.00,0,1,0,...,0,0,0,0,0,0,0,0,0,0


## 1. Exploración: ¿cuándo y quiénes desertan?

In [2]:
por_semestre = df.groupby("semestre").y.agg(["mean", "size"]).rename(columns={"mean": "tasa", "size": "filas"})
por_semestre["tasa"] = (por_semestre.tasa * 100).round(1)
por_semestre.head(10)

,tasa,filas
semestre,,
1,21.9,1491
2,17.9,1231
3,13.3,1011
4,9.0,866
5,7.9,745
6,5.9,660
7,3.5,566
8,4.2,495
9,2.8,426


In [3]:
df["rango_promedio"] = pd.cut(df.promedio, [0, 2.5, 3.0, 3.5, 4.0, 5.0], labels=["< 2,5", "2,5-3,0", "3,0-3,5", "3,5-4,0", "≥ 4,0"])
tabla = df.groupby("rango_promedio", observed=True).y.mean().mul(100).round(1)
ax = tabla.plot.bar(color="#00549F", figsize=(6, 3.5), rot=0)
ax.set_ylabel("% que deserta en el año"); ax.set_xlabel("Promedio del semestre"); ax.set_title("Riesgo según el promedio")
plt.tight_layout(); plt.show()
tabla

rango_promedio
< 2,5      43.7
2,5-3,0    22.4
3,0-3,5    11.8
3,5-4,0     5.2
≥ 4,0       3.2
Name: y, dtype: float64

In [4]:
df["trabaja_mucho"] = np.where(df.horas_trabajo >= 30, "≥ 30 h/semana", np.where(df.horas_trabajo > 0, "1-29 h", "No trabaja"))
df.groupby("trabaja_mucho").y.mean().mul(100).round(1)

trabaja_mucho
1-29 h           18.9
No trabaja        8.0
≥ 30 h/semana    36.5
Name: y, dtype: float64

## 2. Entrenamiento y evaluación

Se separan **estudiantes** (no filas) entre entrenamiento (80 %) y prueba (20 %): si un mismo estudiante apareciera en ambos
lados, el modelo "vería" a la persona y las métricas serían optimistas.

In [5]:
X = df[caracteristicas].to_numpy(float); y = df.y.to_numpy(int); grupos = df.estudiante_id.to_numpy()
ent, prueba = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEMILLA).split(X, y, grupos))
modelo = nuevo_modelo().fit(X[ent], y[ent])
p = modelo.predict_proba(X[prueba])[:, 1]
u = umbrales(y[ent].mean())
m = metricas(y[prueba], p, u["alto"])
print(f"AUC {dec(m['auc'])} · Brier {dec(m['brier'], 4)} · umbrales {u}")
print(f"En ALTO: precisión {pct(m['precisionAlto'])}, sensibilidad {pct(m['sensibilidadAlto'])}")
print("Matriz de confusión:", m["matrizConfusion"])

AUC 0,819 · Brier 0,0915 · umbrales {'medio': 0.11, 'alto': 0.28}
En ALTO: precisión 48,2 %, sensibilidad 47,7 %
Matriz de confusión: {'vp': 95, 'fp': 102, 'vn': 1225, 'fn': 104}


In [6]:
fpr, tpr, _ = roc_curve(y[prueba], p)
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(fpr, tpr, color="#00549F", lw=2, label=f"Modelo (AUC {dec(m['auc'])})"); ax[0].plot([0, 1], [0, 1], "--", color="#888", label="Azar")
ax[0].set_xlabel("Falsos positivos"); ax[0].set_ylabel("Verdaderos positivos"); ax[0].set_title("Curva ROC"); ax[0].legend()
cal = m["calibracion"]
ax[1].plot([c["predicho"] for c in cal], [c["observado"] for c in cal], "o-", color="#00549F", label="Deciles")
ax[1].plot([0, 0.6], [0, 0.6], "--", color="#888", label="Perfecta"); ax[1].set_xlabel("Probabilidad predicha"); ax[1].set_ylabel("Observado")
ax[1].set_title("Calibración"); ax[1].legend(); plt.tight_layout(); plt.show()

In [7]:
aucs = [roc_auc_score(y[v], nuevo_modelo().fit(X[t], y[t]).predict_proba(X[v])[:, 1]) for t, v in GroupKFold(5).split(X, y, grupos)]
print("AUC por pliegue:", [dec(a) for a in aucs], "→ media", dec(np.mean(aucs)))
print("Línea base (solo 'promedio bajo'):", dec(roc_auc_score(y[prueba], -df.promedio.to_numpy()[prueba])))

AUC por pliegue: ['0,855', '0,864', '0,821', '0,805', '0,799'] → media 0,829
Línea base (solo 'promedio bajo'): 0,753


## 3. Interpretación de coeficientes

Las variables están estandarizadas, así que los coeficientes se pueden comparar entre sí. Positivo = aumenta el riesgo.
En la aplicación, el aporte de cada factor para un estudiante es `coeficiente × z`, y la suma de aportes más el intercepto
reconstruye exactamente su probabilidad (explicación exacta, no aproximada).

In [8]:
final = nuevo_modelo().fit(X, y)
coef = pd.Series(final.named_steps["logisticregression"].coef_[0], index=caracteristicas).sort_values()
ax = coef.plot.barh(color=np.where(coef > 0, "#eb6834", "#00549F"), figsize=(7, 6))
ax.axvline(0, color="#333", lw=1); ax.set_title("Coeficientes (naranja aumenta el riesgo)"); plt.tight_layout(); plt.show()

## 4. Exportación

`python ml/entrenar_modelo.py` vuelve a ejecutar estos pasos y escribe `ml/modelo_riesgo.json` y `ml/reporte_modelo.md`.
La prueba `tests/riesgo-modelo.test.ts` verifica que el entrenador TypeScript (usado para reentrenar desde el panel de
administración) reproduce estos mismos coeficientes con el mismo dataset.